In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import kagglehub

# import the kaggle competition dataset
path = kagglehub.competition_download('name-that-llm')
# store in dataframe
df = pd.read_csv(os.path.join(path, "train.csv"), index_col=0)
# print full texts if needed
pd.set_option("display.max_colwidth", None)

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit

# separate data and labels
X = df["text"]
y = df["model"]
# split train and test by prompt
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["prompt_id"]))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# vectorize inputs
vec = TfidfVectorizer()
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)
# apply logistic regression to vectorized input
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, y_train)
# make prediction
y_pred = clf.predict(X_test_vec)
# print test balanced accuracy
print(balanced_accuracy_score(y_test, y_pred))

0.25
